# 04 — Few-shot learning and the label-efficiency curve

**Question this notebook answers:** how many labelled examples per class does it
take to beat zero-shot, and which frozen representation is the most
label-efficient?

This is the chapter that turns a model comparison into the question a manager
actually asks: *how many labels do we have to buy to enter a new problem?*
Everything else in this project exists to make this figure trustworthy.

**Produces:** `figures/04_label_efficiency.png` — the headline figure of the
whole project — plus cached frozen features in `outputs/features/` and the
sweep results in `outputs/results.json`.

**Expected runtime:** ~30 minutes on a Colab T4 (feature extraction dominates;
the probes themselves take seconds).

### Environment setup and persistence

On Colab this clones the repository, installs the pinned dependencies, and — the
part that matters — mounts Google Drive so that **outputs survive the session**.
Locally it is a no-op beyond putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the split and normalisation
statistics, 02 writes the model checkpoint that 04, 05 and 06 all load. Without
persistence, a disconnect means re-running earlier chapters. So `outputs/` and
`figures/` are redirected to Drive via environment variables read by
`s2map.config` at import time.

**`data/` is deliberately NOT on Drive.** The EuroSAT cache is ~2.9 GB and every
training epoch reads it randomly; Drive is a network mount and random reads
through it are slow enough to dominate the runtime. Re-downloading into the
local VM disk each session costs a few minutes and is the faster trade. Set
`USE_DRIVE = False` to keep everything ephemeral.

The install is wrapped so a fragile wheel prints a clear message instead of
killing the kernel halfway through a 40-minute session.

**Re-running this cell picks up code changes.** It hard-resets the clone to
`origin/main` and purges `s2map` from `sys.modules`, because Python caches
imported modules: without the purge, a `git pull` updates the file on disk while
the kernel keeps running the old code, and you get an `AttributeError` for a
function that is visibly there in the source. The clone is treated as a
read-only checkout — edit the notebook in Colab or the code locally, not in
`/content/s2-chips-to-map`, since the reset discards changes there.

**If this cell reports a numpy mismatch**, restart the runtime
(*Runtime → Restart session*) and run it again. Colab preinstalls a mutually
consistent numpy/torch/scipy set; when pip replaces one of them on disk, the
kernel is left holding the old version in memory and every compiled extension
raises `ValueError: numpy.dtype size changed`. Only a restart fixes it. The
requirements file uses compatible *ranges* rather than exact pins for exactly
this group of packages, so it should not happen — the check is there because it
is the single most common way a Colab notebook dies.

In [ ]:
# --- edit these two if you are running your own fork -----------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "s2-chips-to-map"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/s2-chips-to-map")
        for sub in ("outputs", "figures"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        # Read by s2map.config at import time, so this must happen before the import below.
        os.environ["S2MAP_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["S2MAP_FIGURE_DIR"] = str(PERSIST / "figures")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        # Pick up fixes without needing a fresh VM: a stale clone from an earlier
        # session is otherwise invisible and confusing. --depth 1 clones are
        # shallow, so unshallow first or the pull fails on a diverged history.
        subprocess.run(["git", "fetch", "--quiet", "--depth", "50", "origin", "main"], check=False)
        before = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        subprocess.run(["git", "reset", "--hard", "--quiet", "origin/main"], check=False)
        after = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        if before != after:
            print(f"repo updated {before[:7]} -> {after[:7]}")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway — the cells below will report what is missing")

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Drop any already-imported s2map modules before importing them again. Python
# caches modules in sys.modules, so a `git pull` that updates a file on disk has
# NO effect on a kernel that already imported it — you get
# `AttributeError: module 's2map.bands' has no attribute ...` for a function you
# can plainly see in the source. Purging here means re-running this one cell is
# enough to pick up new code, without restarting the runtime and losing state.
_stale = [m for m in list(sys.modules) if m == "s2map" or m.startswith("s2map.")]
for _mod in _stale:
    del sys.modules[_mod]
if _stale:
    print(f"reloaded {len(_stale)} s2map modules from disk (was: {', '.join(sorted(_stale))})")

from s2map import config as C
C.ensure_dirs()
C.print_environment()
print()
# Catches the one Colab failure mode that no amount of re-running fixes:
# pip replaced numpy on disk after this kernel already imported it.
C.check_environment()
cfg = C.load_config()
SEED = C.set_seed(cfg.seed)
print(f"seed             {SEED}   (multi-seed runs use {cfg.seeds})")
DEVICE = C.get_device()
print(f"device           {DEVICE}")
print(f"outputs ->       {C.OUTPUT_DIR}")
print(f"figures ->       {C.FIGURE_DIR}")
print(f"data cache ->    {C.DATA_DIR}  (local/ephemeral by design — see the note above)")
if DEVICE == "cpu":
    print("\n!! no GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.")

### The design

Four frozen encoders, one probing protocol, seven label budgets, five random
draws each.

| Encoder | Input | Fair few-shot competitor? |
|---|---|---|
| CLIP ViT-B/32 | RGB | yes — never saw a EuroSAT label |
| RemoteCLIP ViT-B/32 | RGB | yes — never saw a EuroSAT label |
| ImageNet ResNet-18 | RGB | yes — never saw a EuroSAT label |
| NB02 supervised ResNet-18 | 13-band | **no — it was trained on all 19k training labels** |

That last row is an **upper reference**, not a competitor. It is included
because it answers "how good could a representation of this data possibly be",
and it is drawn dashed and labelled on the plot. Presenting it as one more line
would be the single most misleading thing this project could do, so it is
labelled everywhere it appears.

**Protocol.** For each (encoder, k): draw k labelled chips per class from the
*training* split only, fit a multinomial logistic regression on the frozen
features with the regularisation strength chosen on the validation split, and
evaluate on the same held-out test split every other chapter uses. Repeat with
**5 different random label draws** and report mean ± std.

**Why five draws and not one.** At k = 1, the entire model is determined by which
ten chips you happened to pick, and the spread across draws is enormous — larger
than most of the differences between encoders. A single draw at low k is not a
result, it is an anecdote. This is the methodological point people skip most
often in few-shot papers, and the shaded bands on the headline figure exist to
make it visible.

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
from s2map import bands as B, clip_utils as CL, data as D, evaluate as E, models as M, viz as V

V.set_style()
bundle = D.load_eurosat_ms()
X, y = bundle.X, bundle.y
splits = D.load_splits()
stats = D.load_band_stats()

FEATURE_DIR = C.OUTPUT_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
K_VALUES = list(cfg.few_shot_k)
N_DRAWS = cfg.few_shot_draws
print(f"k values {K_VALUES}   draws per point {N_DRAWS}")
print(f"train pool {splits['train'].size:,}   val {splits['val'].size:,}   test {splits['test'].size:,}")

### Extract and cache the frozen features

Extraction is the slow part of a few-shot study; probing is instant. So every
encoder is run **once** over the whole dataset and cached to disk. That also
means the sweep below can be re-run, extended or re-plotted without touching a
GPU — and that a dead Colab session costs minutes, not the whole notebook.

In [ ]:
def cached(name, fn):
    path = FEATURE_DIR / f"{name}.npz"
    if path.exists():
        with np.load(path) as f:
            print(f"{name:<28} loaded from cache {tuple(f['features'].shape)}")
            return f["features"]
    t0 = time.time()
    feats = fn()
    np.savez_compressed(path, features=feats.astype(np.float32))
    print(f"{name:<28} extracted {tuple(feats.shape)} in {time.time() - t0:.0f}s -> {path.name}")
    return feats

ALL_IDX = np.arange(y.size)   # extract once for every chip; slicing per split is free afterwards

**Encoder 1 and 2 — the CLIP family.** Both consume the same RGB adapter as
notebook 03 (select B04/B03/B02, stretch, resize, normalise), so ten bands are
discarded for both. If RemoteCLIP cannot be loaded, the sweep continues without
it and the figure says so rather than substituting a different model under the
same name.

In [ ]:
def clip_features(loader_name, model, preprocess, chunk=1024):
    out = []
    for s in range(0, ALL_IDX.size, chunk):
        block = np.asarray(X[ALL_IDX[s:s + chunk]])
        images = CL.preprocess_chips(block, preprocess, batch=256)
        out.append(CL.encode_images(model, images, device=DEVICE, batch=256).numpy())
    return np.concatenate(out)

encoders = {}

clip_model, clip_prep, clip_tok = CL.load_openclip("ViT-B-32", "openai", device=DEVICE)
encoders["CLIP ViT-B/32"] = cached("clip_vitb32", lambda: clip_features("clip", clip_model, clip_prep))

try:
    rs_model, rs_prep, rs_tok, _ = CL.load_remoteclip("ViT-B-32", device=DEVICE)
    encoders["RemoteCLIP ViT-B/32"] = cached("remoteclip_vitb32",
                                             lambda: clip_features("rs", rs_model, rs_prep))
except Exception as exc:
    print("!! RemoteCLIP unavailable:", type(exc).__name__, exc)
    print("!! the sweep continues without it; the figure and the README will say so.")

**Encoder 3 — ImageNet ResNet-18.** The "what everyone does by default" arm:
plain supervised ImageNet transfer, RGB, no language involved. It is the control
that tells us whether CLIP's advantage (if any) comes from *language
supervision* or merely from *large-scale pretraining*. Without it, any CLIP win
is uninterpretable.

In [ ]:
# ImageNet ResNet-18, RGB input: the "ordinary transfer learning" reference point.
from torchvision.models import ResNet18_Weights, resnet18

def imagenet_features():
    net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    net.fc = torch.nn.Identity()
    net = net.eval().to(DEVICE)
    # ImageNet statistics, not CLIP's — a mismatched normalisation here would
    # handicap this arm for a reason that has nothing to do with the encoder.
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    out = []
    with torch.no_grad():
        for s in range(0, ALL_IDX.size, 512):
            block = np.asarray(X[ALL_IDX[s:s + 512]])
            # same RGB adapter CLIP gets: stretch to 8-bit-equivalent [0,1], then resize
            rgb = np.stack([np.transpose(B.true_color(c), (2, 0, 1)) for c in block])
            t = torch.from_numpy(rgb).float()
            t = torch.nn.functional.interpolate(t, size=(224, 224), mode="bicubic", align_corners=False)
            t = (t - mean) / std
            out.append(net(t.to(DEVICE)).float().cpu().numpy())
    return np.concatenate(out)

encoders["ImageNet ResNet-18"] = cached("imagenet_resnet18", imagenet_features)

**Encoder 4 — the notebook 02 supervised backbone, as an upper reference.** This
one saw all 18,900 training labels, so a "10-shot probe" on its features is not
10-shot anything. It is included to answer a different question — how separable
this data is *in principle*, given a representation built specifically for it —
and it is drawn dashed and explicitly labelled on the figure. Silently plotting
it alongside the honest arms would be the single most misleading thing this
project could do.

In [ ]:
# The NB02 supervised backbone. UPPER REFERENCE ONLY — it saw all training labels.
ckpt_path = C.OUTPUT_DIR / "nb02_supervised_backbone_reference.pt"
REFERENCE_KEY = "NB02 supervised 13-band"

if ckpt_path.exists():
    def supervised_features():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        net = M.build_resnet18(13, 10, pretrained=False)
        net.load_state_dict(ckpt["state_dict"])
        net = net.eval().to(DEVICE)
        out = []
        with torch.no_grad():
            for s in range(0, ALL_IDX.size, 512):
                block = D.normalize(np.asarray(X[ALL_IDX[s:s + 512]]), stats)
                t = torch.from_numpy(block).to(DEVICE)
                out.append(net.forward_features(t).float().cpu().numpy())
        return np.concatenate(out)

    encoders[REFERENCE_KEY] = cached("nb02_supervised_13band", supervised_features)
else:
    print(f"!! {ckpt_path.name} not found — run notebook 02 first to include the reference arm.")

print("\nencoders available:", list(encoders))
for name, f in encoders.items():
    print(f"  {name:<26}{f.shape}")

### The probe

Multinomial logistic regression on L2-normalised frozen features.

**Why a linear probe** rather than fine-tuning, as the primary protocol: it
measures the *representation*, not the optimisation. If two encoders are probed
with the same classifier under the same regularisation search, the difference
between them is a property of their features. It is also convex, so there is no
seed-to-seed optimisation noise contaminating the label-draw variance we are
trying to measure.

**Regularisation is tuned on validation, per (encoder, k).** At k = 1 the right
C is far smaller than at k = 100 — the amount of regularisation a 10-sample
problem needs is not the amount a 1000-sample problem needs, and fixing C would
quietly handicap one end of the curve. Tuning on the test set instead would
invalidate the whole figure.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

C_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

def probe(features, train_idx, val_idx, test_idx, labels):
    Ftr = normalize(features[train_idx])
    best_c, best_val = None, -1.0
    for c in C_GRID:
        clf = LogisticRegression(C=c, max_iter=2000, n_jobs=-1)
        clf.fit(Ftr, labels[train_idx])
        score = E.macro_f1(labels[val_idx], clf.predict(normalize(features[val_idx])), 10)
        if score > best_val:
            best_c, best_val = c, score
    clf = LogisticRegression(C=best_c, max_iter=2000, n_jobs=-1)
    clf.fit(Ftr, labels[train_idx])
    pred = clf.predict(normalize(features[test_idx]))
    return {
        "macro_f1": E.macro_f1(labels[test_idx], pred, 10),
        "accuracy": E.accuracy(labels[test_idx], pred),
        "C": best_c, "val_macro_f1": best_val,
    }

### The sweep

k ∈ {1, 2, 5, 10, 20, 50, 100} labels per class × 5 draws × every encoder.
Around 2,000 logistic regressions in total, which is fast because the features
are cached — this is the payoff of separating extraction from probing.

Note that with k=100 and ten classes, the probe still sees only 1,000 of the
18,900 available training labels.

In [ ]:
curves = {}
t0 = time.time()
for name, feats in encoders.items():
    means, stds, per_k = [], [], {}
    for k in K_VALUES:
        scores = []
        for draw in range(N_DRAWS):
            idx = D.few_shot_indices(y, splits["train"], k=k, seed=1000 * draw + k)
            scores.append(probe(feats, idx, splits["val"], splits["test"], y)["macro_f1"])
        means.append(float(np.mean(scores)))
        stds.append(float(np.std(scores)))
        per_k[k] = scores
        print(f"{name:<26} k={k:<4} macro-F1 {means[-1]:.4f} ± {stds[-1]:.4f}")
    curves[name] = {"k": K_VALUES, "mean": means, "std": stds, "raw": per_k,
                    "reference": name == REFERENCE_KEY}
    print()
print(f"sweep finished in {time.time() - t0:.0f}s")

**Look at the k=1 standard deviations before reading anything else.** They are
typically several times larger than the gaps between encoders at that budget.
Any paper reporting a single 1-shot number is reporting noise, and the shaded
bands in the figure below are there so a reader cannot miss this.

### The comparison arm: full fine-tuning at the same budgets

A linear probe is not the only option at low k — the alternative is to fine-tune
the whole network on those same k×10 examples.

**The expected crossover.** At very low k, fine-tuning 11M parameters on 10–50
examples overfits catastrophically and the frozen-feature probe wins. Past some
budget, the extra capacity starts to pay and fine-tuning overtakes. Locating
that crossover point is the practically useful output: it tells you *which
recipe to use* at your label budget, which is a more actionable answer than
"which model is best".

Run at a subset of k values and fewer draws, for runtime — stated here rather
than hidden.

In [ ]:
from s2map import train as T, transforms as TR

FT_K = [5, 20, 100]
FT_DRAWS = 3
ft_cfg = C.TrainConfig(epochs=15, batch_size=32, lr=1e-4, patience=6,
                       num_workers=cfg.train.num_workers, amp=cfg.train.amp)

ft_means, ft_stds = [], []
for k in FT_K:
    scores = []
    for draw in range(FT_DRAWS):
        C.set_seed(draw)
        idx = D.few_shot_indices(y, splits["train"], k=k, seed=1000 * draw + k)
        ds_tr = D.EuroSATChips(X, y, idx, stats, TR.train_transform(), list(B.RGB_INDICES))
        ds_va = D.EuroSATChips(X, y, splits["val"], stats, None, list(B.RGB_INDICES))
        ds_te = D.EuroSATChips(X, y, splits["test"], stats, None, list(B.RGB_INDICES))
        model = M.build_resnet18(3, 10, pretrained=True)
        model, hist = T.fit(model, D.make_loader(ds_tr, min(32, k * 10), True, 0, draw),
                            D.make_loader(ds_va, 256, False, cfg.train.num_workers, draw),
                            ft_cfg, DEVICE, verbose=False)
        logits, labels = T.predict(model, D.make_loader(ds_te, 256, False, cfg.train.num_workers, draw), DEVICE)
        scores.append(E.macro_f1(labels, logits.argmax(1), 10))
    ft_means.append(float(np.mean(scores)))
    ft_stds.append(float(np.std(scores)))
    print(f"fine-tuned ResNet-18 (RGB)  k={k:<4} macro-F1 {ft_means[-1]:.4f} ± {ft_stds[-1]:.4f}")

curves["Fine-tuned ResNet-18 (RGB)"] = {"k": FT_K, "mean": ft_means, "std": ft_stds,
                                        "reference": False}

Locate the crossover: the smallest budget at which fine-tuning beats the best
frozen probe. Interpolating on log k, because that is the axis the curve is
smooth in and the sweep points are log-spaced.

In [ ]:
# Where does fine-tuning overtake the best frozen probe? Interpolate on log k.
probe_names = [n for n, c in curves.items() if not c["reference"] and n != "Fine-tuned ResNet-18 (RGB)"]
crossover = None
for k, m in zip(FT_K, ft_means):
    best_probe = max(np.interp(np.log(k), np.log(curves[n]["k"]), curves[n]["mean"]) for n in probe_names)
    print(f"k={k:<4} fine-tune {m:.4f}   best frozen probe {best_probe:.4f}   "
          f"{'fine-tuning ahead' if m > best_probe else 'probe ahead'}")
    if crossover is None and m > best_probe:
        crossover = k
print("\ncrossover (first k where fine-tuning overtakes the best frozen probe):", crossover)

### The headline figure

x: labels per class, log scale. y: test macro-F1. One line per encoder with
±1 std bands. Dashed horizontal line: notebook 02's full-supervision ceiling
(18,900 labels). Dotted horizontal lines: notebook 03's zero-shot results
(0 labels). The reference encoder is drawn dashed and labelled.

Read left to right, the figure answers: *what does each additional order of
magnitude of labelling budget actually buy?*

In [ ]:
results_all = E.load_results()

def lookup(notebook, arm, key="test_macro_f1"):
    for r in results_all:
        if r.get("notebook") == notebook and r.get("arm") == arm:
            v = r.get(key)
            return v["mean"] if isinstance(v, dict) else v
    return None

ceiling = lookup("02", "_ceiling")
zero_shot = {}
for arm, label in (("clip_vitb32_v4_prompt_ensemble", "CLIP"),
                   ("remoteclip_vitb32", "RemoteCLIP"),
                   ("laion_clip_vitb32_fallback", "LAION CLIP (fallback)")):
    v = lookup("03", arm)
    if v is not None:
        zero_shot[label] = v
print("ceiling from NB02:", ceiling, " zero-shot anchors from NB03:", zero_shot)

fig = V.plot_label_efficiency(curves, zero_shot=zero_shot, supervised_ceiling=ceiling)
ax = fig.axes[0]
if crossover is not None:
    yc = float(np.interp(np.log(crossover), np.log(FT_K), ft_means))
    V.annotate_crossing(ax, crossover, yc, f"fine-tuning overtakes\nfrozen probes at k≈{crossover}")
V.save(fig, "04_label_efficiency")

### The sentence

The deliverable of this project is not a number, it is a sentence a
decision-maker can act on. The cell below composes it from the measured results
so it cannot drift from what was actually run.

In [ ]:
def labels_to_beat(target, curve):
    """Smallest k in the sweep whose mean macro-F1 exceeds `target`."""
    for k, m in zip(curve["k"], curve["mean"]):
        if m >= target:
            return k
    return None

lines = []
for zs_name, zs_value in zero_shot.items():
    for enc in probe_names:
        k_needed = labels_to_beat(zs_value, curves[enc])
        lines.append(f"  a linear probe on {enc} needs {k_needed if k_needed else '>' + str(K_VALUES[-1])} "
                     f"labels/class to match {zs_name} zero-shot ({zs_value:.3f})")

best_at_max = max(probe_names, key=lambda n: curves[n]["mean"][-1])
gap = (ceiling - curves[best_at_max]["mean"][-1]) if ceiling else float("nan")

print("\n".join(lines))
print(f"\nAt k={K_VALUES[-1]}, the best frozen probe ({best_at_max}) reaches "
      f"{curves[best_at_max]['mean'][-1]:.3f} macro-F1, still {gap:.3f} short of the "
      f"full-supervision ceiling of {ceiling:.3f} trained on {splits['train'].size:,} labels.")
print(f"Fine-tuning overtakes every frozen representation at roughly k={crossover} labels per class.")

E.append_result({
    "notebook": "04", "arm": "label_efficiency_sweep",
    "input": "frozen features + logistic probe",
    "k_values": K_VALUES, "draws_per_point": N_DRAWS,
    "curves": {n: {"k": c["k"], "mean": c["mean"], "std": c["std"], "reference": c["reference"]}
               for n, c in curves.items()},
    "crossover_k_finetune_beats_probe": crossover,
    "supervised_ceiling_macro_f1": ceiling,
    "zero_shot_anchors": zero_shot,
    "notes": "the NB02 13-band encoder is an upper reference: it saw all training labels",
})
print("\nresults.json updated")

### Optional: a k-NN probe as a hyperparameter-free second protocol

A linear probe has one knob (C) that we tuned. A nearest-centroid / k-NN probe
has effectively none, so if the two protocols rank the encoders the same way,
the ranking is a property of the representations rather than of the probe. If
they disagree, that is the more interesting outcome and worth chasing.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print(f"{'encoder':<28}{'k=5 linear':>12}{'k=5 kNN':>10}{'k=50 linear':>13}{'k=50 kNN':>10}")
knn_rows = {}
for name, feats in encoders.items():
    row = []
    for k in (5, 50):
        scores = []
        for draw in range(3):
            idx = D.few_shot_indices(y, splits["train"], k=k, seed=1000 * draw + k)
            clf = KNeighborsClassifier(n_neighbors=min(5, k), metric="cosine")
            clf.fit(normalize(feats[idx]), y[idx])
            scores.append(E.macro_f1(y[splits["test"]], clf.predict(normalize(feats[splits["test"]])), 10))
        row.append(float(np.mean(scores)))
    knn_rows[name] = row
    lin5 = curves[name]["mean"][curves[name]["k"].index(5)]
    lin50 = curves[name]["mean"][curves[name]["k"].index(50)]
    print(f"{name:<28}{lin5:>12.3f}{row[0]:>10.3f}{lin50:>13.3f}{row[1]:>10.3f}")

rank_lin = sorted(encoders, key=lambda n: -curves[n]["mean"][curves[n]["k"].index(50)])
rank_knn = sorted(encoders, key=lambda n: -knn_rows[n][1])
print("\nranking at k=50, linear probe:", rank_lin)
print("ranking at k=50, kNN probe:   ", rank_knn)
print("protocols agree on the ranking:", rank_lin == rank_knn)

## Findings

_Numbers from the run; the claims are the point._

1. **The label-efficiency curve is the deliverable.** It converts "which model is
   best" into "what does a labelling budget buy", which is the form of the
   question that gets asked in practice.

2. **Domain-specific pretraining shows up as label efficiency, not just
   accuracy.** RemoteCLIP's advantage is largest at small k and narrows as
   labels accumulate — the value of good pretraining is precisely that it
   substitutes for labels.

3. **1-shot results are dominated by draw variance.** The k=1 standard deviations
   are several times the gaps between encoders. Reporting a single 1-shot number
   would be reporting the draw, not the model.

4. **Frozen probes beat fine-tuning at low k, and lose at high k**, crossing over
   at the annotated point. Below it, fine-tuning 11M parameters on a few dozen
   examples overfits; above it, the extra capacity pays.

5. **Even the best frozen probe at k=100 does not reach the 13-band supervised
   ceiling** — the residual gap is the ten discarded bands plus 18,000 labels.

6. **The upper-reference encoder is not a competitor** and is labelled as such
   wherever it appears. It answers a different question: how separable this data
   is in principle.

**Next:** notebook 05 takes the best supervised model out of the curated
benchmark and onto a real Sentinel-2 scene, where the domain shift bites.